In [ ]:
# ---- LangSmith evaluation setup ----
# Load env vars (LANGSMITH_API_KEY etc.) and enable LangSmith tracing
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACKING"] = "true"

#Create a data point for evaluation
from langsmith import Client
client = Client()

# Define the dataset name and create a new dataset using the LangSmith client
dataset_name = "LangSmith Evaluation Dataset Final"
dataset = client.create_dataset(dataset_name)

# Ground-truth Q&A examples derived from the meeting notes
examples=[
  # --- Engineering meetings (file 1) ---
  {
    "input": "Who was responsible for reviewing the Jetpack Compose migration?",
    "output": "Michael Brown, the Android Tech Lead, was assigned to review the Compose migration progress in the Android Application Performance Review meeting."
  },
  {
    "input": "What decisions were made in the Database Optimization Discussion meeting?",
    "output": "The team decided to add indexes to frequently searched columns, archive old transactional data, and enable database query monitoring."
  },
  {
    "input": "Which meeting discussed cloud infrastructure optimization and what were the decisions?",
    "output": "The Cloud Infrastructure Review meeting (July 6, 2026) decided to remove unused environments, enable automatic scaling, and review monthly cloud expenses."
  },
  {
    "input": "What improvements were planned for the Android application?",
    "output": "The team planned to migrate remaining screens to Jetpack Compose, improve image caching implementation, and reduce unnecessary network requests."
  },
  {
    "input": "Who was assigned to optimize SQL queries?",
    "output": "Sarah Lee was assigned to optimize the SQL queries in the Database Optimization Discussion meeting."
  },
  {
    "input": "What decisions were made in the Security Review Meeting?",
    "output": "The team decided to enable additional API rate limiting and improve authentication logging."
  },
  {
    "input": "What was decided during the API Integration Planning meeting?",
    "output": "The team decided that API integration will happen in two phases, sandbox testing must complete before production deployment, and additional monitoring will be added."
  },
  {
    "input": "What was decided in the Mobile Release Readiness meeting?",
    "output": "The release candidate will be submitted after final QA, and crash monitoring will remain enabled after launch."
  },
  {
    "input": "What action items were assigned to David Wilson?",
    "output": "David Wilson was assigned to prepare the migration plan in the Backend Architecture Review meeting."
  }
]

# Upload examples to the LangSmith dataset (inputs use "question" key, outputs use "answer")
client.create_examples(
    inputs=[
        {"question": ex["input"]}
        for ex in examples
    ],
    outputs=[
        {"answer": ex["output"]}
        for ex in examples
    ],
    dataset_id=dataset.id,
)

In [ ]:
# ---- Judge LLM for evaluation ----
# Re-use the same model config as the main app for consistency
from dotenv import load_dotenv
from pathlib import Path
import json
from langchain_openai import ChatOpenAI


BASE_DIR = Path.cwd()
load_dotenv(BASE_DIR / ".env")

# Model name used by both the target app and the judge
model = os.getenv("LLM_MODEL", "gpt-4o-mini")
print(f"Using model: {model}", flush=True)

# ChatOpenAI instance used as the judge to score correctness / faithfulness
judge_llm = ChatOpenAI(
    model=model,
    api_key=os.getenv("OMNIROUTER_API_KEY"),
    base_url=os.getenv("OMNIROUTER_BASE_URL"),
    streaming=True,
)

In [ ]:
# ---- OpenAI-compatible client ----
# Wraps the raw OpenAI client with LangSmith tracing so evaluation runs are logged
import openai
from langsmith import wrappers

openrouter_client = openai.OpenAI(
    api_key=os.getenv("OMNIROUTER_API_KEY"),
    base_url=os.getenv("OMNIROUTER_BASE_URL")
)
 
# LangSmith wrapper that automatically traces LLM calls made through this client
openai_client=wrappers.wrap_openai(openrouter_client)


In [ ]:
# ---- System prompt for the correctness judge ----
# The judge LLM compares the AI answer to a reference answer and decides
# whether the core response is factually correct (CORRECT vs INCORRECT).
eval_prompt_system = """
You are evaluating answer correctness.

Compare the AI answer with the reference answer.

Rules:
- Focus on whether the AI correctly answers the user's question.
- The AI answer does NOT need to match the reference exactly.
- Different wording and additional correct details are acceptable.
- Do not penalize the AI for omitting secondary details.
- Mark INCORRECT only if:
  - it contains incorrect facts,
  - it misses the main answer, or
  - it contradicts the reference.

Output exactly:
CORRECT
or
INCORRECT
"""

In [ ]:
# ---- Correctness evaluator ----
# Calls the judge LLM to compare the AI response against the reference answer.
# Returns True when the judge deems the answer factually correct.
def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:

    prompt = f"""
        Question:
        {inputs['question']}

        Reference Answer:
        {reference_outputs['answer']}

        AI Answer:
        {outputs['response']}

        Evaluate whether the AI answer correctly answers the question.

        Rules:
        - Focus on factual correctness, not exact wording.
        - The AI answer does not need to include every detail from the reference answer.
        - Do not mark incorrect if the answer is shorter but contains the correct information.
        - Mark incorrect only if it contains wrong information or misses the core answer.

        Return exactly:
        CORRECT
        or
        INCORRECT
"""

    result = judge_llm.invoke(
        [
            ("system", eval_prompt_system),
            ("human", prompt)
        ]
    )

    return result.content.strip().upper() == "CORRECT"

In [ ]:
# ---- Faithfulness evaluation prompts ----
# These prompts ask the judge LLM to verify that every claim in the AI
# response is grounded in the retrieved source documents (no hallucinations).
faithfulness_system_prompt = """
You are a faithfulness evaluator for AI-generated meeting notes.

Your task:
Determine whether the AI response is fully grounded in the retrieved source documents.

Rules:
- Retrieved source documents are the ONLY ground truth.
- The reference answer is only a brief summary; do not use it to judge faithfulness.
- Extra details are acceptable if they are supported by the source documents.
- Do not penalize names, dates, decisions, action items, or meeting details that exist in the source documents.
- Mark a claim as hallucinated only when it is not supported by any retrieved source document.
- If all claims are supported, return FAITHFUL.
- If any claim is unsupported, return UNFAITHFUL.

Output exactly one word:
FAITHFUL or UNFAITHFUL
"""

faithfulness_user_prompt = """
Retrieved Source Documents:
{source_documents}

Reference Answer (secondary context only):
{reference_answer}

AI Generated Response:
{generated_answer}

Evaluation Instructions:
- Check each factual claim in the AI response against the retrieved source documents.
- A claim is supported if it is explicitly stated or strongly supported by the documents.
- Do not reject additional details that are present in the documents, even if missing from the reference answer.
- Do not use the reference answer as the source of truth.
- Do not accept unsupported assumptions, guesses, or external knowledge.

Decision:
Return FAITHFUL if every claim is supported.
Return UNFAITHFUL if any claim is unsupported.

Output exactly one word:
FAITHFUL or UNFAITHFUL
"""

In [ ]:
# ---- Faithfulness evaluator ----
# Retrieves the same Chroma documents the app would use, then asks the judge LLM
# whether the AI response is fully supported by those source documents.
def faithfulness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:

    question = inputs["question"]

    # Pull the top-5 similar docs from Chroma for this question
    source_documents = query_documents.invoke({"question": question})

    prompt = faithfulness_user_prompt.format(
        source_documents=source_documents,
        reference_answer=reference_outputs["answer"],
        generated_answer=outputs["response"]
    )

    result = judge_llm.invoke(
        [
            ("system", faithfulness_system_prompt),
            ("human", prompt)
        ]
    )

    verdict = result.content.strip().upper()

    print(f"Faithfulness Verdict: {verdict}", flush=True)

    return verdict == "FAITHFUL"

In [ ]:
# ---- Target application under evaluation ----
# Simplified version of main.py: retrieves meeting-note context from Chroma
# and asks the LLM to answer using only those documents.
from tools import query_documents



def my_app(question: str) -> str:
    """
    End-to-end RAG pipeline: retrieve documents → build prompt → invoke LLM.
    Returns the assistant's answer as a plain string.
    """

    app_llm = ChatOpenAI(
        model=model,
        api_key=os.getenv("OMNIROUTER_API_KEY"),
        base_url=os.getenv("OMNIROUTER_BASE_URL"),
        streaming=True,
    )

    # Retrieve relevant meeting notes from Chroma
    context = query_documents.invoke({
        "question": question
    })

    prompt = f"""
Use ONLY the following meeting notes.

Meeting Notes:
{context}

Question:
{question}

If the answer is not contained in the meeting notes,
say "I couldn't find that information."
"""

    try:
        result = app_llm.invoke(
            [
                ("system", "You answer questions from meeting notes."),
                ("human", prompt)
            ]
        )
    except ValueError:
        # Fallback: retry without streaming if the stream fails
        app_llm_fallback = ChatOpenAI(
            model=model,
            api_key=os.getenv("OMNIROUTER_API_KEY"),
            base_url=os.getenv("OMNIROUTER_BASE_URL"),
            streaming=False,
        )
        result = app_llm_fallback.invoke(
            [
                ("system", "You answer questions from meeting notes."),
                ("human", prompt)
            ]
        )

    return result.content.strip()

In [ ]:
# ---- LangSmith target wrapper ----
# Wraps my_app in the {response} dict format that LangSmith expects
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [ ]:
# ---- Run the full evaluation ----
# Evaluates every example in the dataset against both correctness and
# faithfulness evaluators. Results are logged to LangSmith under the
# given experiment prefix.
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,faithfulness],
    experiment_prefix="open-router-meeting-notes-chatbot"
)